# Financial Statement Quality & Earnings Surprise Model

**Vicente Pozo | Academic empirical research | Real-data execution, 20 October 2026**

Developed with AI assistance. This notebook studies the association between accounting-quality measures,
historical earnings surprises and subsequent benchmark-adjusted equity returns. It does not implement an investable portfolio.

## tl;dr

**Executed sample:** 300 matched announcements from 15 companies; 2021-01-15 through 2025-11-20. All 300 matched events have 21-session outcomes.

**Holdout:** 225 training events after purge and 74 test events beginning 2024-10-16.

**Prediction check:** Random Forest RMSE = 5.72 percentage points; training-mean baseline = 5.51 pp. Forest error is higher; holdout R² = -0.082.

**Earnings surprise:** 21-session Q5−Q1 spread -0.13 pp (quarter-block bootstrap 95% interval -2.55 to +2.33 pp; Q5 n=48, Q1 n=66).

**Accounting quality:** 21-session Q5−Q1 spread -0.54 pp (quarter-block bootstrap 95% interval -2.42 to +1.27 pp; Q5 n=58, Q1 n=56).

**Composite:** 21-session Q5−Q1 spread -0.44 pp (quarter-block bootstrap 95% interval -3.03 to +1.59 pp; Q5 n=57, Q1 n=62).

**Interpretation:** findings describe this retrospective sample. Provider estimate vintages, survivor selection, sector differences and dependent observations prevent stronger forecasting or investment-performance claims.

## Context & Methods
Fixed convenience sample: the original 15 companies. Announcements: **2021-01-01 through 2025-12-31**.
SEC filings: 2019–2025, including returned historical submission files. Prices: 2020-01-01 through
2026-02-13 (end-exclusive request: 2026-02-15), providing a warm-up and forward-return buffer.

### Key assumptions
- Only filings dated strictly before the earnings announcement enter that event's quality signal.
- Entry is the first trading close strictly after the New York announcement date; horizons are 1, 5 and 21 sessions.
- Composite weights (50/50) and the 20-percentage-point surprise scale are fixed before observing results.
- The last 25% of entry dates are held out; training labels reaching the test cutoff are purged.
- The universe contains JPM, whose accounting is not fully comparable with nonfinancial companies. A prespecified sensitivity excludes JPM.
- Current-vintage historical estimates, corporate-action adjustments and a survivor convenience sample limit predictive interpretation.

## Data
SEC EDGAR supplies filed fundamentals and filing dates. Yahoo Finance supplies adjusted prices and historical EPS estimates/reported EPS through yfinance.
Downloads are cached with retrieval timestamps and hashes. Raw vendor records remain outside the public repository.

In [1]:
import os, sys, time, json, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from IPython.display import display, Markdown

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))
from src.data_sources import (get_json, filing_history, market_prices, earnings_events,
                              DEFAULT_SEC_USER_AGENT, SourceAccessError)
from src.research_core import (numeric, fact_table, choose_ttm_flow, choose_balance,
                               compute_quality, match_events, attach_forward_returns,
                               assign_history_buckets, chronological_holdout, event_profile)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', 22)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
BLUE, GOLD, CHARCOAL = '#245982', '#AD7922', '#353B43'
plt.rcParams.update({'figure.figsize':(10,5), 'font.size':11, 'axes.spines.top':False,
                     'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':.18,
                     'axes.prop_cycle':plt.cycler(color=[BLUE,GOLD,CHARCOAL])})
SEC_USER_AGENT = os.environ.get('SEC_USER_AGENT', DEFAULT_SEC_USER_AGENT)
CACHE_DIR = ROOT_DIR/'data/real_snapshot_20261019'
OUTPUT_DIR = ROOT_DIR/'outputs'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
UNIVERSE = ['AAPL','MSFT','AMZN','GOOGL','META','NVDA','JPM','JNJ','PG','KO','XOM','CVX','WMT','HD','CAT']
BENCHMARK = '^GSPC'
START_DATE, END_DATE = '2020-01-01', '2026-02-15'
EVENT_START, EVENT_END = '2021-01-01', '2026-01-01'
FILING_START, FILING_END = '2019-01-01', '2026-01-01'
SEC_TICKERS_URL = 'https://www.sec.gov/files/company_tickers.json'
SEC_FACTS_URL = 'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
SEC_SUBMISSIONS_URL = 'https://data.sec.gov/submissions/CIK{cik}.json'
print(f'Locked scope: {len(UNIVERSE)} original issuers; 2021–2025 announcements; 21-session primary horizon.')

Locked scope: 15 original issuers; 2021–2025 announcements; 21-session primary horizon.


### 1. Identify companies and preserve source snapshots
The SEC request identifies this public academic project by its repository URL. An optional personal
contact setting is read from the environment and is never committed. Downloads stop on access denial
or rate limiting. Existing source snapshots are reused unchanged; delete a snapshot explicitly to refresh it.

In [2]:
tickers = get_json(SEC_TICKERS_URL, CACHE_DIR/'sec_company_tickers.json', SEC_USER_AGENT)
SEC_MAP = {str(v['ticker']).upper(): {'cik':str(v['cik_str']).zfill(10), 'title':v['title']}
           for v in tickers.values() if v.get('ticker')}
assert all(t in SEC_MAP for t in UNIVERSE), 'An original issuer has no current SEC mapping.'
HISTORICAL_MAPPING={'XOM':{'cik':'0000034088','title':'Exxon Mobil Corporation (pre-July 2026 registrant)',
 'reason':'2021–2025 study predates the July 2026 holding-company reorganization',
 'source':'https://www.sec.gov/Archives/edgar/data/2115436/000119312526291990/d71068d8k12b.htm'}}
for ticker,item in HISTORICAL_MAPPING.items():
 SEC_MAP[ticker]={'cik':item['cik'],'title':item['title']}
(OUTPUT_DIR/'historical_entity_mapping.json').write_text(json.dumps(HISTORICAL_MAPPING,indent=2))
display(pd.DataFrame([{'ticker':t,**SEC_MAP[t]} for t in UNIVERSE]))

,ticker,cik,title
0,AAPL,0000320193,Apple Inc.
1,MSFT,0000789019,MICROSOFT CORP
2,AMZN,0001018724,AMAZON COM INC
3,GOOGL,0001652044,Alphabet Inc.
4,META,0001326801,"Meta Platforms, Inc."
5,NVDA,0001045810,NVIDIA CORP
6,JPM,0000019617,JPMORGAN CHASE & CO
7,JNJ,0000200406,JOHNSON & JOHNSON
8,PG,0000080424,PROCTER & GAMBLE Co
9,KO,0000021344,COCA COLA CO


### 2. Load adjusted market prices
Adjusted stock closes and the S&P 500 price index use identical trading dates. Their difference is a
benchmark-adjusted holding-period return, not factor-model alpha. Index dividends and stock total-return
adjustments are not perfectly matched; this limitation is retained in interpretation.

In [3]:
prices = pd.concat([market_prices(t, START_DATE, END_DATE, CACHE_DIR)
                    for t in UNIVERSE+[BENCHMARK]], axis=1).sort_index()
available = [t for t in UNIVERSE if t in prices.columns]
assert len(available)==len(UNIVERSE) and BENCHMARK in prices, 'Incomplete required market universe.'
assert not prices.index.duplicated().any()
price_coverage = prices.agg(['count','min','max']).T
price_coverage['first_date']=[prices[c].first_valid_index() for c in prices]
price_coverage['last_date']=[prices[c].last_valid_index() for c in prices]
display(price_coverage[['count','first_date','last_date']])
normalized = prices[UNIVERSE].div(prices[UNIVERSE].iloc[0]).mul(100)
fig,ax=plt.subplots(figsize=(11,6))
for t in UNIVERSE:
 ax.plot(normalized.index,normalized[t],linewidth=1.1,label=t)
ax.set(title='Adjusted stock-price paths | fixed 15-company sample',ylabel='Index (first close = 100)',xlabel='Date')
ax.legend(ncol=5,fontsize=9,loc='upper left')
plt.tight_layout();plt.show()

,count,first_date,last_date
AAPL,"1,538.0000",2020-01-02,2026-02-13
MSFT,"1,538.0000",2020-01-02,2026-02-13
AMZN,"1,538.0000",2020-01-02,2026-02-13
GOOGL,"1,538.0000",2020-01-02,2026-02-13
META,"1,538.0000",2020-01-02,2026-02-13
NVDA,"1,538.0000",2020-01-02,2026-02-13
JPM,"1,538.0000",2020-01-02,2026-02-13
JNJ,"1,538.0000",2020-01-02,2026-02-13
PG,"1,538.0000",2020-01-02,2026-02-13
KO,"1,538.0000",2020-01-02,2026-02-13


Figure 1: saved below as an external image.

![Figure 1](../docs/results/figure_01.png)

### 3. Construct accounting observations from eligible filings
Flows use comparable trailing-twelve-month periods, with original filing dates retained. SEC historical
submission files are fetched only when their returned date range intersects 2019–2025. Missing compatible
facts stay missing. The data-quality table exposes the resulting coverage.

In [4]:
FACT_TAGS = {
    "revenue": [
        ("us-gaap","RevenueFromContractWithCustomerExcludingAssessedTax"),
        ("us-gaap","Revenues"),
        ("us-gaap","SalesRevenueNet"),
    ],
    "net_income": [("us-gaap","NetIncomeLoss")],
    "cfo": [("us-gaap","NetCashProvidedByUsedInOperatingActivities")],
    "gross_profit": [("us-gaap","GrossProfit")],
    "assets": [("us-gaap","Assets")],
    "liabilities": [("us-gaap","Liabilities")],
    "accounts_receivable": [
        ("us-gaap","AccountsReceivableNetCurrent"),
        ("us-gaap","AccountsReceivableCurrent"),
    ],
    "inventory": [("us-gaap","InventoryNet"), ("us-gaap","InventoryGross")],
}


FACT_TAGS['consolidated_profit']=[('us-gaap','ProfitLoss')]
FACT_TAGS['noncontrolling_income']=[('us-gaap','NetIncomeLossAttributableToNoncontrollingInterest')]

def build_filings(ticker):
    cik=SEC_MAP[ticker]['cik']
    facts=get_json(SEC_FACTS_URL.format(cik=cik),CACHE_DIR/f'{ticker}_{cik}_facts.json',SEC_USER_AGENT)
    submissions=get_json(SEC_SUBMISSIONS_URL.format(cik=cik),CACHE_DIR/f'{ticker}_{cik}_submissions.json',SEC_USER_AGENT)
    history=filing_history(cik,submissions,CACHE_DIR,FILING_START,FILING_END,SEC_USER_AGENT)
    tables={name:fact_table(facts,name,FACT_TAGS) for name in FACT_TAGS}
    rows=[]
    for _,filing in history.iterrows():
        fd,rd=filing['filingDate'],filing['reportDate']
        prior=rd-pd.DateOffset(years=1)
        row={'ticker':ticker,'company':SEC_MAP[ticker]['title'],'filing_date':fd,
             'report_date':rd,'form':filing['form'],'accession_number':filing['accessionNumber']}
        for name in ['revenue','net_income','cfo','gross_profit']:
            row[name]=choose_ttm_flow(tables[name],fd,rd)
        row['net_income_basis']='direct parent net income'
        if not np.isfinite(row['net_income']):
            consolidated=choose_ttm_flow(tables['consolidated_profit'],fd,rd)
            noncontrolling=choose_ttm_flow(tables['noncontrolling_income'],fd,rd)
            if np.isfinite(consolidated) and np.isfinite(noncontrolling):
                row['net_income']=consolidated-noncontrolling
                row['net_income_basis']='consolidated profit less noncontrolling income'
            else:
                row['net_income_basis']='unavailable; no substitution'
        for name in ['assets','liabilities','accounts_receivable','inventory']:
            row[name]=choose_balance(tables[name],fd,rd)
        row['prior_year_assets']=choose_balance(tables['assets'],fd,prior)
        row['prior_year_revenue']=choose_ttm_flow(tables['revenue'],fd,prior)
        prior_gp=choose_ttm_flow(tables['gross_profit'],fd,prior)
        row['prior_year_gross_margin']=(prior_gp/row['prior_year_revenue']
               if np.isfinite(row['prior_year_revenue']) and row['prior_year_revenue']!=0 else np.nan)
        rows.append(row)
    result=pd.DataFrame(rows).drop_duplicates(['ticker','filing_date'],keep='last')
    print(ticker, 'eligible filings:',len(result),flush=True)
    return result

sec=pd.concat([build_filings(t) for t in UNIVERSE],ignore_index=True)
assert not sec.empty
assert set(sec['ticker'])==set(UNIVERSE), 'An original issuer has no eligible historical filings.'
assert sec['filing_date'].lt(FILING_END).all()
display(sec.groupby('ticker').agg(filings=('filing_date','count'),first_filing=('filing_date','min'),last_filing=('filing_date','max')))

AAPL eligible filings: 28
MSFT eligible filings: 28
AMZN eligible filings: 28
GOOGL eligible filings: 28
META eligible filings: 28
NVDA eligible filings: 28
JPM eligible filings: 28
JNJ eligible filings: 28
PG eligible filings: 28
KO eligible filings: 28
XOM eligible filings: 28
CVX eligible filings: 28
WMT eligible filings: 28
HD eligible filings: 28
CAT eligible filings: 28


,filings,first_filing,last_filing
ticker,,,
AAPL,28,2019-01-30,2025-10-31
AMZN,28,2019-02-01,2025-10-31
CAT,28,2019-02-14,2025-11-03
CVX,28,2019-02-22,2025-11-06
GOOGL,28,2019-02-05,2025-10-30
HD,28,2019-03-28,2025-11-25
JNJ,28,2019-02-20,2025-10-22
JPM,28,2019-02-26,2025-11-04
KO,28,2019-02-21,2025-10-23


### 4. Compute the fixed accounting-quality score
Quality combines cash conversion, accrual intensity, ROAA, margin change, revenue growth and leverage.
Ranks use the latest known observation for each company at that filing date. At least five peers and
50% of absolute feature weight are required. Different sectors are not inherently comparable; the
prespecified nonfinancial sensitivity is reported later.

In [5]:
sec=compute_quality(sec)
feature_cols=['cfo_ni_ratio','accrual_ratio','roaa','gross_margin_change','revenue_growth','leverage']
coverage=sec.groupby('ticker')[feature_cols+['quality_score']].count()
coverage.insert(0,'filings',sec.groupby('ticker').size())
display(coverage)
assert sec['quality_score'].notna().any()

,filings,cfo_ni_ratio,accrual_ratio,roaa,gross_margin_change,revenue_growth,leverage,quality_score
ticker,,,,,,,,
AAPL,28,28,28,28,28,28,28,27
AMZN,28,27,28,28,0,28,0,28
CAT,28,24,24,28,0,28,28,24
CVX,28,24,28,28,0,28,28,28
GOOGL,28,28,28,28,0,28,28,28
HD,28,28,28,28,28,28,28,28
JNJ,28,28,28,28,27,27,28,28
JPM,28,28,28,28,0,7,28,28
KO,28,26,26,28,26,26,0,26


In [6]:
fig,axes=plt.subplots(1,2,figsize=(15,5.8))
sns.boxplot(data=sec.dropna(subset=['quality_score']),x='ticker',y='quality_score',
            color='#CFDDEA',ax=axes[0])
axes[0].set(title='Accounting quality | 2019–2025 filings',ylabel='Weighted rank score',xlabel='Issuer')
axes[0].tick_params(axis='x',rotation=50)
corr_cols=['quality_score']+feature_cols
sns.heatmap(sec[corr_cols].corr(),annot=True,fmt='.2f',cmap='RdBu_r',center=0,
            vmin=-1,vmax=1,ax=axes[1],cbar_kws={'label':'Pearson correlation'})
axes[1].set_title('Accounting feature correlations | descriptive')
plt.tight_layout();plt.show()

Figure 2: saved below as an external image.

![Figure 2](../docs/results/figure_02.png)

### 5. Load earnings announcements and compute surprises
Surprise = `100 × (reported EPS − estimated EPS) / abs(estimated EPS)`. Zero or missing estimates
remain missing. Dates use New York time. Future announcements and announcements outside 2021–2025
are excluded before linking outcomes. Historical EPS estimates come from the provider's current
history; their exact pre-announcement vintage cannot be independently certified.

In [7]:
def earnings_history(ticker):
    frame=earnings_events(ticker,CACHE_DIR).reset_index()
    frame=frame.rename(columns={frame.columns[0]:'earnings_datetime','EPS Estimate':'eps_estimate','Reported EPS':'reported_eps'})
    frame['earnings_datetime']=pd.to_datetime(frame['earnings_datetime'],utc=True).dt.tz_convert('America/New_York')
    frame['earnings_date']=frame['earnings_datetime'].dt.tz_localize(None).dt.normalize()
    for col in ['eps_estimate','reported_eps']:
        frame[col]=numeric(frame[col])
    frame['surprise_pct']=100*(frame['reported_eps']-frame['eps_estimate'])/frame['eps_estimate'].abs().replace(0,np.nan)
    frame['ticker']=ticker
    return frame[['ticker','earnings_datetime','earnings_date','eps_estimate','reported_eps','surprise_pct']]
all_earnings=pd.concat([earnings_history(t) for t in UNIVERSE],ignore_index=True)
all_earnings=all_earnings.drop_duplicates(['ticker','earnings_datetime'])
candidate_earnings=all_earnings[all_earnings['earnings_date'].ge(EVENT_START)&all_earnings['earnings_date'].lt(EVENT_END)].copy()
earnings=candidate_earnings.dropna(subset=['surprise_pct']).copy()
assert not earnings.empty
earnings_coverage=earnings.groupby('ticker').agg(events=('earnings_date','size'),first=('earnings_date','min'),last=('earnings_date','max'))
display(earnings_coverage)
fig,ax=plt.subplots()
ax.hist(earnings['surprise_pct'],bins=40,color=BLUE,edgecolor='white')
ax.set(title='Observed historical EPS surprises | 2021–2025',xlabel='EPS surprise (%)',ylabel='Announcements')
plt.tight_layout();plt.show()

,events,first,last
ticker,,,
AAPL,20,2021-01-27,2025-10-30
AMZN,20,2021-02-02,2025-10-30
CAT,20,2021-01-29,2025-10-29
CVX,20,2021-01-29,2025-10-31
GOOGL,20,2021-02-02,2025-10-29
HD,20,2021-02-23,2025-11-18
JNJ,20,2021-01-26,2025-10-14
JPM,20,2021-01-15,2025-10-14
KO,20,2021-02-10,2025-10-21


Figure 3: saved below as an external image.

![Figure 3](../docs/results/figure_03.png)

### 6. Match information available before each announcement
Only filings strictly earlier than the announcement date are eligible. An event is omitted when
its latest eligible filing has no usable quality score. The composite weights are fixed at 50/50.

In [8]:
events=match_events(earnings,sec)
assert not events.empty and events['filing_precedes_earnings'].all()
assert events['earnings_date'].ge(EVENT_START).all() and events['earnings_date'].lt(EVENT_END).all()
display(events.groupby('ticker').agg(matched_events=('earnings_date','size'),first=('earnings_date','min'),last=('earnings_date','max')))

,matched_events,first,last
ticker,,,
AAPL,20,2021-01-27,2025-10-30
AMZN,20,2021-02-02,2025-10-30
CAT,20,2021-01-29,2025-10-29
CVX,20,2021-01-29,2025-10-31
GOOGL,20,2021-02-02,2025-10-29
HD,20,2021-02-23,2025-11-18
JNJ,20,2021-01-26,2025-10-14
JPM,20,2021-01-15,2025-10-14
KO,20,2021-02-10,2025-10-21


## Results
### 7. Measure forward benchmark-adjusted returns
Entry occurs at the first benchmark trading close strictly after the announcement. Horizons require
h+1 prices and use identical stock/index dates. Missing observations remain missing; no zero returns are substituted.

In [9]:
events=attach_forward_returns(events,prices,BENCHMARK)
for horizon in [1,5,21]:
 valid=events[f'abnormal_return_{horizon}d'].notna()
 assert events.loc[valid,f'entry_{horizon}d'].gt(events.loc[valid,'earnings_date']).all()
 assert events.loc[valid,f'exit_{horizon}d'].gt(events.loc[valid,f'entry_{horizon}d']).all()
display(pd.DataFrame({'horizon_sessions':[1,5,21],
                     'complete_events':[events[f'abnormal_return_{h}d'].notna().sum() for h in [1,5,21]]}))

,horizon_sessions,complete_events
0,1,300
1,5,300
2,21,300


### 8. Compare fixed signals and trailing-history buckets
Cutoffs use events in the preceding 365 days, excluding the current day, with at least 20 prior events.
Q5−Q1 is a difference of historical bucket means, not a simultaneous long/short portfolio return.
Intervals resample calendar-quarter blocks (2,000 draws, fixed seed), preserving within-quarter dependence.
These intervals have limited precision with only twenty quarters and do not eliminate issuer dependence.

In [10]:
SIGNALS={'Earnings surprise':('surprise_signal','surprise_quintile'),
         'Accounting quality':('quality_score','quality_quintile'),
         'Composite':('composite_score','composite_quintile')}
for value,bucket in [('surprise_pct','surprise_quintile'),('quality_score','quality_quintile'),('composite_score','composite_quintile')]:
 events[bucket]=assign_history_buckets(events,value)
events['calendar_quarter']=events['earnings_date'].dt.to_period('Q').astype(str)

def block_interval(frame,bucket,target='abnormal_return_21d',draws=2000):
    usable=frame.dropna(subset=[target,bucket])
    blocks=sorted(usable['calendar_quarter'].unique())
    if len(blocks)<4:return np.nan,np.nan
    stats_by_block=[]
    for block in blocks:
        sub=usable[usable['calendar_quarter']==block]
        high=sub.loc[sub[bucket]==5,target];low=sub.loc[sub[bucket]==1,target]
        stats_by_block.append([high.sum(),len(high),low.sum(),len(low)])
    values=np.array(stats_by_block,dtype=float)
    rng=np.random.default_rng(RANDOM_SEED)
    totals=values[rng.integers(0,len(blocks),size=(draws,len(blocks)))].sum(axis=1)
    good=(totals[:,1]>0)&(totals[:,3]>0)
    spreads=totals[good,0]/totals[good,1]-totals[good,2]/totals[good,3]
    return tuple(np.quantile(spreads,[.025,.975]))

def signal_scorecard(frame):
    rows=[]
    for label,(value,bucket) in SIGNALS.items():
        sub=frame.dropna(subset=['abnormal_return_21d',bucket])
        high=sub.loc[sub[bucket]==5,'abnormal_return_21d'];low=sub.loc[sub[bucket]==1,'abnormal_return_21d']
        lo,hi=block_interval(frame,bucket)
        rows.append({'signal':label,'high_n':len(high),'low_n':len(low),
                     'high_mean':high.mean(),'low_mean':low.mean(),
                     'q5_minus_q1':high.mean()-low.mean(),'ci95_low':lo,'ci95_high':hi})
    return pd.DataFrame(rows)
scorecard=signal_scorecard(events)
display(scorecard.style.format({c:'{:.2%}' for c in ['high_mean','low_mean','q5_minus_q1','ci95_low','ci95_high']}))
fig,ax=plt.subplots(figsize=(10,4.5))
y=np.arange(len(scorecard))
ax.hlines(y,scorecard['ci95_low']*100,scorecard['ci95_high']*100,color=BLUE,lw=2)
ax.scatter(scorecard['q5_minus_q1']*100,y,color=BLUE,s=55,zorder=3)
ax.axvline(0,color=CHARCOAL,lw=1)
ax.set_yticks(y,scorecard['signal']);ax.invert_yaxis()
ax.set(title='21-session bucket spread | quarter-block bootstrap 95% intervals',xlabel='Q5 minus Q1 mean adjusted return (percentage points)')
plt.tight_layout();plt.show()
correlation_rows=[]
for label,(value,_) in SIGNALS.items():
 for horizon in [1,5,21]:
    sub=events[[value,f'abnormal_return_{horizon}d']].dropna()
    correlation_rows.append({'signal':label,'horizon':horizon,'n':len(sub),
             'Pearson':sub.iloc[:,0].corr(sub.iloc[:,1]),
             'Spearman':sub.iloc[:,0].corr(sub.iloc[:,1],method='spearman')})
correlations=pd.DataFrame(correlation_rows)
display(correlations)

,signal,high_n,low_n,high_mean,low_mean,q5_minus_q1,ci95_low,ci95_high
0,Earnings surprise,48,66,0.25%,0.38%,-0.13%,-2.55%,2.33%
1,Accounting quality,58,56,-0.18%,0.36%,-0.54%,-2.42%,1.27%
2,Composite,57,62,-0.25%,0.19%,-0.44%,-3.03%,1.59%


Figure 4: saved below as an external image.

,signal,horizon,n,Pearson,Spearman
0,Earnings surprise,1,300,0.0738,0.0567
1,Earnings surprise,5,300,0.0801,0.0309
2,Earnings surprise,21,300,0.0082,0.0161
3,Accounting quality,1,300,-0.0854,-0.1238
4,Accounting quality,5,300,-0.1028,-0.1160
5,Accounting quality,21,300,-0.0518,-0.0775
6,Composite,1,300,-0.0042,-0.0400
7,Composite,5,300,-0.0114,-0.0432
8,Composite,21,300,-0.0287,-0.0488


![Figure 4](../docs/results/figure_04.png)

### 9. Evaluate a chronological holdout
The last 25% of unique entry dates form the test set. Training labels overlapping that boundary are
removed. Imputation and model fitting use training data only. Random Forest parameters are fixed;
there is no hyperparameter search on the holdout. OLS uses issuer-clustered standard errors for
exploratory inference; fifteen clusters and overlapping events still limit precision.

In [11]:
model_features=['surprise_signal','quality_score','accrual_ratio','roaa','gross_margin_change','revenue_growth','leverage']
train,test=chronological_holdout(events)
assert len(train)>=40 and len(test)>=10, 'Insufficient post-purge events for the declared model evaluation.'
assert train['exit_21d'].max()<test['entry_21d'].min()
X_train=train[model_features].replace([np.inf,-np.inf],np.nan)
X_test=test[model_features].replace([np.inf,-np.inf],np.nan)
usable=X_train.columns[X_train.notna().any()]
X_train,X_test=X_train[usable],X_test[usable]
y_train,y_test=train['abnormal_return_21d'],test['abnormal_return_21d']
imputer=SimpleImputer(strategy='median')
design=pd.DataFrame(imputer.fit_transform(X_train),columns=usable,index=X_train.index)
ols=sm.OLS(y_train,sm.add_constant(design,has_constant='add')).fit(
    cov_type='cluster',cov_kwds={'groups':train['ticker']})
display(pd.DataFrame({'coefficient':ols.params,'clustered_SE':ols.bse,'exploratory_p':ols.pvalues}))
forest=make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(
    n_estimators=400,max_depth=4,min_samples_leaf=8,random_state=RANDOM_SEED,n_jobs=1))
forest.fit(X_train,y_train)
predicted=forest.predict(X_test)
baseline_pred=np.full(len(test),float(y_train.mean()))
def evaluate_model(label,pred):
    residual=pred-y_test.to_numpy()
    return {'model':label,'test_n':len(test),'RMSE':float(np.sqrt(np.mean(residual**2))),
            'MAE':float(np.mean(np.abs(residual))),
            'test_R2':float(1-np.sum(residual**2)/np.sum((y_test-y_test.mean())**2))}
model_results=pd.DataFrame([evaluate_model('Random Forest',predicted),evaluate_model('Training-mean baseline',baseline_pred)])
display(model_results)
print('Training events:',len(train),'Test events:',len(test),'Test starts:',test['entry_21d'].min().date())
print('Latest training label:',train['exit_21d'].max().date())
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
axes[0].bar(model_results['model'],model_results['RMSE']*100,color=[BLUE,GOLD])
axes[0].set(title='Chronological holdout prediction error',ylabel='RMSE (percentage points)')
axes[0].tick_params(axis='x',rotation=12)
axes[1].scatter(y_test*100,predicted*100,color=BLUE,alpha=.65,edgecolors='white')
bound=max(np.abs(y_test).max(),np.abs(predicted).max())*100
axes[1].plot([-bound,bound],[-bound,bound],color=CHARCOAL,ls='--',lw=1)
axes[1].set(title=f'Actual vs predicted | {len(test)} held-out events',xlabel='Actual adjusted return (%)',ylabel='Predicted adjusted return (%)')
plt.tight_layout();plt.show()

Training events: 225 Test events: 74 Test starts: 2024-10-16
Latest training label: 2024-09-30


,coefficient,clustered_SE,exploratory_p
const,-0.0116,0.0185,0.5293
surprise_signal,-0.0030,0.0119,0.7992
quality_score,-0.0484,0.0248,0.0510
accrual_ratio,-0.3222,0.1527,0.0349
roaa,-0.0113,0.0833,0.8918
gross_margin_change,0.2577,0.1096,0.0187
revenue_growth,0.0338,0.0176,0.0542
leverage,-0.0074,0.0291,0.8006


,model,test_n,RMSE,MAE,test_R2
0,Random Forest,74,0.0572,0.0464,-0.0819
1,Training-mean baseline,74,0.0551,0.0441,-0.0054


Figure 5: saved below as an external image.

![Figure 5](../docs/results/figure_05.png)

### 10. Prespecified sector sensitivity and event paths
The nonfinancial sensitivity removes JPM before rebuilding quality ranks, composite scores and historical
bucket thresholds. It does not reuse full-universe ranks. It is a comparability check, not a new optimized signal.
Event paths show additive cumulative daily stock-minus-index returns and differ from compounded holding-period spreads.

In [12]:
raw_sec=sec.drop(columns=['quality_score','quality_coverage','quality_peer_count'])
nonfinancial_sec=compute_quality(raw_sec[raw_sec['ticker']!='JPM'].copy())
nonfinancial_events=match_events(earnings[earnings['ticker']!='JPM'].copy(),nonfinancial_sec)
nonfinancial_events=attach_forward_returns(nonfinancial_events,prices,BENCHMARK)
for value,bucket in [('surprise_pct','surprise_quintile'),('quality_score','quality_quintile'),('composite_score','composite_quintile')]:
 nonfinancial_events[bucket]=assign_history_buckets(nonfinancial_events,value)
nonfinancial_events['calendar_quarter']=nonfinancial_events['earnings_date'].dt.to_period('Q').astype(str)
sensitivity=signal_scorecard(nonfinancial_events)
sensitivity['sample']='Nonfinancial only (ranks rebuilt)'
display(sensitivity.style.format({c:'{:.2%}' for c in ['high_mean','low_mean','q5_minus_q1','ci95_low','ci95_high']}))
fig,ax=plt.subplots()
for bucket,label,color in [(1,'Low composite',GOLD),(5,'High composite',BLUE)]:
 profile=event_profile(events,prices,bucket,BENCHMARK)
 if not profile.empty:ax.plot(profile.index,profile*100,label=label,color=color,lw=2)
ax.set(title='Additive daily event profile | historical composite buckets',xlabel='Trading sessions after entry close',ylabel='Mean cumulative daily adjusted return (%)')
ax.legend();plt.tight_layout();plt.show()

,signal,high_n,low_n,high_mean,low_mean,q5_minus_q1,ci95_low,ci95_high,sample
0,Earnings surprise,45,61,0.78%,0.60%,0.18%,-3.05%,3.14%,Nonfinancial only (ranks rebuilt)
1,Accounting quality,54,53,-0.03%,-0.11%,0.08%,-1.90%,1.99%,Nonfinancial only (ranks rebuilt)
2,Composite,52,62,-0.08%,0.64%,-0.72%,-3.53%,1.63%,Nonfinancial only (ranks rebuilt)


Figure 6: saved below as an external image.

![Figure 6](../docs/results/figure_06.png)

### 11. Coverage, reproducibility and independent calculation checks
The next cell reconciles source coverage, validates a sample of filing facts and returns independently,
and records exact source hashes. Full event-level vendor data remain local, excluded from public GitHub.

In [13]:
coverage_rows=[]
for t in UNIVERSE:
 candidate=candidate_earnings[candidate_earnings['ticker']==t]
 usable_events=earnings[earnings['ticker']==t]
 matched=events[events['ticker']==t]
 coverage_rows.append({'ticker':t,'candidate_announcements':len(candidate),
  'numeric_surprises':len(usable_events),'matched_quality':len(matched),
  'complete_21d':int(matched['abnormal_return_21d'].notna().sum()),
  'eligible_filings':int((sec['ticker']==t).sum())})
data_coverage=pd.DataFrame(coverage_rows)
display(data_coverage)
checks=[]
for _,r in events.dropna(subset=['abnormal_return_21d']).sample(min(10,len(events)),random_state=RANDOM_SEED).iterrows():
 entry,exit=r['entry_21d'],r['exit_21d']
 independent=(float(prices.at[exit,r['ticker']])/float(prices.at[entry,r['ticker']])-1)-(
  float(prices.at[exit,BENCHMARK])/float(prices.at[entry,BENCHMARK])-1)
 checks.append(bool(np.isclose(independent,r['abnormal_return_21d'],atol=1e-12)))
assert all(checks)
# Reconcile a published full-year Apple observation to its actual accession, not to a later restatement.
facts=json.loads((CACHE_DIR/f"AAPL_{SEC_MAP['AAPL']['cik']}_facts.json").read_text())
annual=sec[(sec['ticker']=='AAPL')&(sec['form']=='10-K')].sort_values('filing_date').iloc[-1]
fact_checks=[]
for metric in ['revenue','net_income','cfo']:
 table=fact_table(facts,metric,FACT_TAGS)
 raw=table[(table['accn']==annual['accession_number'])&table['end'].eq(annual['report_date'])]
 duration=(raw['end']-raw['start']).dt.days
 raw=raw[duration.between(300,430)]
 assert not raw.empty, f'No accession-specific annual audit row for {metric}'
 actual=float(raw.sort_values('tag_priority').iloc[0]['value'])
 assert np.isclose(actual,annual[metric]), f'Annual accounting mismatch for {metric}'
 fact_checks.append({'ticker':'AAPL','metric':metric,'report_date':str(annual['report_date'].date()),
                     'accession_number':annual['accession_number'],'agrees':True})
print(f'Independent checks: {sum(checks)}/{len(checks)} event returns; {len(fact_checks)} annual filing inputs.')
manifest=[]
for meta_path in sorted(CACHE_DIR.glob('*.meta.json')):
 record=json.loads(meta_path.read_text());record['file']=meta_path.name.removesuffix('.meta.json')
 manifest.append(record)
versions={}
from importlib.metadata import version
for package in ['numpy','pandas','requests','yfinance','statsmodels','scikit-learn','scipy','matplotlib','seaborn','nbformat','nbconvert','IPython']:
 versions[package]=version(package)
summary={'status':'executed real-data retrospective academic study',
 'execution_utc':pd.Timestamp.now(tz='UTC').isoformat(),'universe':UNIVERSE,
 'event_start':EVENT_START,'event_end_exclusive':EVENT_END,'price_start':START_DATE,
 'price_end_exclusive':END_DATE,'benchmark':BENCHMARK,'filing_rows':len(sec),
 'candidate_events':len(candidate_earnings),'numeric_surprises':len(earnings),
 'matched_events':len(events),'complete_21d':int(events['abnormal_return_21d'].notna().sum()),
 'event_first':str(events['earnings_date'].min().date()),'event_last':str(events['earnings_date'].max().date()),
 'train_events':len(train),'test_events':len(test),'test_first_entry':str(test['entry_21d'].min().date()),
 'last_train_exit':str(train['exit_21d'].max().date()),'random_seed':RANDOM_SEED,
 'return_checks_passed':sum(checks),'annual_fact_checks':fact_checks,'package_versions':versions,
 'limitations':['fixed survivor convenience sample','historical estimate vintage unverified',
 'current provider adjustments and revisions','sector comparability','overlapping events and few inference clusters',
 'gross observational returns; no implemented portfolio or cost analysis']}
for frame,name in [(sec,'sec_filing_quality_panel'),(events,'earnings_quality_event_panel'),
                    (earnings,'earnings_events_raw'),(nonfinancial_events,'nonfinancial_event_panel')]:
 frame.to_csv(OUTPUT_DIR/f'{name}.csv',index=False)
for frame,name in [(data_coverage,'data_coverage'),(scorecard,'research_scorecard'),
                    (correlations,'signal_correlations'),(model_results,'model_results'),(sensitivity,'sector_sensitivity')]:
 frame.to_csv(OUTPUT_DIR/f'{name}.csv',index=False)
(OUTPUT_DIR/'run_metadata.json').write_text(json.dumps(summary,indent=2))
(OUTPUT_DIR/'source_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved derived outputs, provenance and validation metadata.')

Independent checks: 10/10 event returns; 3 annual filing inputs.
Saved derived outputs, provenance and validation metadata.


,ticker,candidate_announcements,numeric_surprises,matched_quality,complete_21d,eligible_filings
0,AAPL,20,20,20,20,28
1,MSFT,20,20,20,20,28
2,AMZN,20,20,20,20,28
3,GOOGL,20,20,20,20,28
4,META,20,20,20,20,28
5,NVDA,20,20,20,20,28
6,JPM,20,20,20,20,28
7,JNJ,20,20,20,20,28
8,PG,20,20,20,20,28
9,KO,20,20,20,20,28


## Takeaways
The following cell generates the interpretation from executed values. A weak association or a model
that fails to beat its baseline is a valid research finding. The results do not establish causation,
tradable alpha, an unbiased historical investment universe or verified pre-announcement consensus vintages.

In [14]:
rf=model_results.iloc[0];base=model_results.iloc[1]
direction='lower' if rf['RMSE']<base['RMSE'] else 'higher'
lines=[f"**Executed sample:** {len(events):,} matched announcements from {events['ticker'].nunique()} companies; "
       f"{summary['event_first']} through {summary['event_last']}. All {summary['complete_21d']:,} matched events have 21-session outcomes.",
       f"**Holdout:** {len(train)} training events after purge and {len(test)} test events beginning {summary['test_first_entry']}.",
       f"**Prediction check:** Random Forest RMSE = {rf['RMSE']*100:.2f} percentage points; "
       f"training-mean baseline = {base['RMSE']*100:.2f} pp. Forest error is {direction}; "
       f"holdout R² = {rf['test_R2']:.3f}."]
for _,row in scorecard.iterrows():
 lines.append(f"**{row['signal']}:** 21-session Q5−Q1 spread {row['q5_minus_q1']*100:+.2f} pp "
              f"(quarter-block bootstrap 95% interval {row['ci95_low']*100:+.2f} to {row['ci95_high']*100:+.2f} pp; "
              f"Q5 n={int(row['high_n'])}, Q1 n={int(row['low_n'])}).")
lines.append('**Interpretation:** findings describe this retrospective sample. Provider estimate vintages, survivor selection, '
             'sector differences and dependent observations prevent stronger forecasting or investment-performance claims.')
display(Markdown('\n\n'.join(lines)))
_=(OUTPUT_DIR/'RESULTS.md').write_text('# Executed research results\n\n'+'\n\n'.join(lines)+'\n')

**Executed sample:** 300 matched announcements from 15 companies; 2021-01-15 through 2025-11-20. All 300 matched events have 21-session outcomes.

**Holdout:** 225 training events after purge and 74 test events beginning 2024-10-16.

**Prediction check:** Random Forest RMSE = 5.72 percentage points; training-mean baseline = 5.51 pp. Forest error is higher; holdout R² = -0.082.

**Earnings surprise:** 21-session Q5−Q1 spread -0.13 pp (quarter-block bootstrap 95% interval -2.55 to +2.33 pp; Q5 n=48, Q1 n=66).

**Accounting quality:** 21-session Q5−Q1 spread -0.54 pp (quarter-block bootstrap 95% interval -2.42 to +1.27 pp; Q5 n=58, Q1 n=56).

**Composite:** 21-session Q5−Q1 spread -0.44 pp (quarter-block bootstrap 95% interval -3.03 to +1.59 pp; Q5 n=57, Q1 n=62).

**Interpretation:** findings describe this retrospective sample. Provider estimate vintages, survivor selection, sector differences and dependent observations prevent stronger forecasting or investment-performance claims.